In [10]:
# ── 셀 1 │ 설정 ──
import os, sys, json, time, random
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import add_self_loops, softmax
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split, StratifiedKFold
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

NEW = r'C:\Users\iisl\Desktop\0_Molecualar_Informatics\2026_final\2026_new' + '\\'
OUT = NEW + r'03_dedup\\'
sys.path.insert(0, NEW + '01_pre-train')
import featurizer as FZ

DEV   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = (42, 43, 44)
FT_LR_BB, FT_LR_CLF, FT_EPOCHS, FT_PAT, FT_BS = 1e-4, 5e-3, 60, 12, 32
FT_BS_EV = 64         # 기존 파이프라인 BS_EV

for arm in ['keep', 'drop']:
    p = OUT + f'pretrain_{arm}.pt'
    assert os.path.exists(p), f'없음: {p} — 01_dedup_pretrain 먼저 돌리세요'
print(DEV, '| 체크포인트 OK')

cuda | 체크포인트 OK


In [11]:
# ── 셀 2 │ GAT_SI ──
class GH(nn.Module):
    def __init__(s, ic, oc, d=0.3):
        super().__init__()
        s.W = nn.Parameter(torch.empty(ic, oc))
        s.a = nn.Parameter(torch.empty(2 * oc, 1))
        nn.init.xavier_uniform_(s.W); nn.init.xavier_uniform_(s.a)
        s.lr = nn.LeakyReLU(0.2); s.d = d
    def forward(s, x, ei):
        ei, _ = add_self_loops(ei, num_nodes=x.size(0))
        h = x @ s.W
        row, col = ei
        e = s.lr((torch.cat([h[row], h[col]], -1) @ s.a).squeeze(-1))
        e = F.dropout(e, s.d, training=s.training)
        al = softmax(e, row)
        o = torch.zeros_like(h)
        o.index_add_(0, row, h[col] * al.unsqueeze(-1))
        return F.elu(o)

class GL(nn.Module):
    def __init__(s, ic, oc, h, d=0.3):
        super().__init__()
        s.h, s.d = h, d
        s.att = nn.ModuleList([GH(ic, oc, d) for _ in range(h)])
    def forward(s, x, ei):
        if s.h == 1:
            z = torch.mean(torch.stack([a(x, ei) for a in s.att], 0), 0)
        else:
            z = torch.cat([a(x, ei) for a in s.att], -1)
        return F.dropout(z, s.d, training=s.training)

class AttentionPool(nn.Module):
    def __init__(s, ch, d=0.3):
        super().__init__()
        s.gate = nn.Sequential(nn.Linear(ch, ch), nn.ReLU(),
                               nn.Dropout(d), nn.Linear(ch, 1))
    def forward(s, x, batch):
        al = softmax(s.gate(x).squeeze(-1), batch)
        n = int(batch.max()) + 1
        out = torch.zeros(n, x.size(1), device=x.device, dtype=x.dtype)
        out.index_add_(0, batch, x * al.unsqueeze(-1))
        return out

class GAT_SI(nn.Module):
    def __init__(s, ic, hc=32, h=8, d=0.3):
        super().__init__()
        s.l1 = GL(ic, hc, h, d)
        s.l2 = GL(hc * h, hc, h, d)
        s.l3 = GL(hc * h, hc, 1, d)
        s.pool = AttentionPool(hc, d)
        s.clf = nn.Sequential(nn.Linear(hc, 64), nn.ReLU(),
                              nn.Dropout(d), nn.Linear(64, 1))
    def backbone_parameters(s):
        return (list(s.l1.parameters()) + list(s.l2.parameters())
                + list(s.l3.parameters()) + list(s.pool.parameters()))
    def forward(s, x, ei, b):
        x = F.elu(s.l1(x, ei)); x = F.elu(s.l2(x, ei)); x = s.l3(x, ei)
        return s.clf(s.pool(x, b)).squeeze(-1)

In [13]:
# ── 셀 3 │ 임상 그래프 ──
def _prep(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try: Chem.SanitizeMol(m)
    except Exception: return None
    try: Chem.AssignStereochemistry(m, cleanIt=True, force=True)
    except Exception: pass
    return m

def make_graph(smi, y=None):
    mol = _prep(smi)
    if mol is None: return None
    try: x = np.stack([FZ.atom_features(a) for a in mol.GetAtoms()])
    except Exception: return None
    src, dst, ea = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = FZ.bond_features(b)
        src += [i, j]; dst += [j, i]; ea += [bf, bf]
    if not src: return None
    g = Data(x=torch.tensor(np.asarray(x, np.float32)),
             edge_index=torch.tensor([src, dst], dtype=torch.long),
             edge_attr=torch.tensor(np.asarray(ea, np.float32)))
    if y is not None:
        g.y = torch.tensor([float(y)], dtype=torch.float)
    return g

def scaffold_of(smi):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(smiles=smi,
                                                   includeChirality=False)
    except Exception:
        return smi

CACHE = OUT + 'graphs_clin.pt'
if os.path.exists(CACHE):
    blob = torch.load(CACHE, weights_only=False)
    Cg, Cm = blob['graphs'], blob['meta']
    print('캐시 로드', len(Cg))
else:
    clin = pd.read_csv(NEW + r'data\processed\clinical\clinical_pool_endpoints.csv',
                       encoding='utf-8-sig')
    # canon_smiles / scaffold 는 전처리 노트북이 이미 만들어 뒀다
    clin = clin.rename(columns={'canon_smiles': 'smiles'})
    if 'scaffold' not in clin.columns:
        clin['scaffold'] = clin['smiles'].map(scaffold_of)
    KEEP = ['key14','smiles','y','scaffold','endpoint_A','endpoint_B','listed']
    KEEP = [c for c in KEEP if c in clin.columns]
    Cg, rec = [], []
    for _, r in clin.iterrows():
        g = make_graph(r['smiles'], r['y'])
        if g is None: continue
        Cg.append(g)
        rec.append({c: r[c] for c in KEEP})
    Cm = pd.DataFrame(rec)
    torch.save({'graphs': Cg, 'meta': Cm}, CACHE)
    print('생성', len(Cg), '/', len(clin))

print(len(Cg), dict(Cm['y'].value_counts()),
      '| scaffold', Cm['scaffold'].nunique())

생성 1225 / 1225
1225 {1: 725, 0: 500} | scaffold 835


In [14]:
# ── 셀 4 │ metrics / fit ──
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def metrics(y, p, thr=0.5):
    yh = (np.asarray(p) >= thr).astype(int); y = np.asarray(y).astype(int)
    return dict(accuracy=accuracy_score(y, yh),
                auroc=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
                aucpr=average_precision_score(y, p) if len(set(y)) > 1 else np.nan,
                f1=f1_score(y, yh, zero_division=0),
                precision=precision_score(y, yh, zero_division=0),
                recall=recall_score(y, yh, zero_division=0))

@torch.no_grad()
def predict(model, loader):
    model.eval(); ps, ys = [], []
    for b in loader:
        b = b.to(DEV)
        ps += torch.sigmoid(model(b.x, b.edge_index, b.batch)).cpu().tolist()
        ys += b.y.view(-1).cpu().tolist()
    return np.array(ys), np.array(ps)

def fit(model, tr_ds, va_ds, opt, epochs, patience, bsz):
    tr = DataLoader(tr_ds, batch_size=bsz, shuffle=True)
    vl = DataLoader(va_ds, batch_size=FT_BS_EV)
    npos = sum(int(g.y.item()) for g in tr_ds); nneg = len(tr_ds) - npos
    pw = torch.tensor([max(nneg,1)/max(npos,1)], dtype=torch.float).to(DEV)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    best, state, w = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        for b in tr:
            b = b.to(DEV); opt.zero_grad()
            crit(model(b.x, b.edge_index, b.batch),
                 b.y.view(-1).float()).backward()
            opt.step()
        ys, ps = predict(model, vl)
        a = roc_auc_score(ys, ps) if len(set(ys.astype(int))) > 1 else 0.5
        if a > best:
            best, w = a, 0
            state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            w += 1
            if w >= patience: break
    if state:
        model.load_state_dict({k: v.to(DEV) for k, v in state.items()})
    return model, best

In [21]:
def finetune_once(train_idx, test_idx, yv, ckpt, seed):
    set_seed(seed)
    m = GAT_SI(IC).to(DEV)
    if ckpt is not None:                       # None 이면 clinical-only(scratch)
        sd = torch.load(ckpt, map_location=DEV, weights_only=True)
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert all(k.startswith('clf.') for k in missing), missing
        assert not unexpected, unexpected
        m.clf.apply(_reset)                    # classifier 재초기화
    tr, va = train_test_split(train_idx, test_size=0.10,
                              random_state=seed, stratify=yv[train_idx])
    def ds(ix):
        out = []
        for i in ix:
            g = Cg[i].clone()
            g.y = torch.tensor([float(yv[i])], dtype=torch.float)
            out.append(g)
        return out
    opt = torch.optim.Adam(
        [{'params': m.backbone_parameters(), 'lr': FT_LR_BB},
         {'params': m.clf.parameters(),      'lr': FT_LR_CLF}],
        weight_decay=5e-4)
    m, _ = fit(m, ds(tr), ds(va), opt, FT_EPOCHS, FT_PAT, FT_BS)
    ys, ps = predict(m, DataLoader(ds(test_idx), batch_size=FT_BS_EV))
    return metrics(ys, ps)

In [22]:
# ── 셀 6 │ dedup ablation CV (Broad, scaffold) ──
yv, scaf = Cm['y'].astype(int).values, Cm['scaffold'].values

FOLDS = {sd: scaffold_folds(scaf, sd) for sd in SEEDS}

CONDS = [('Clinical only',    'Before', None),                        # 1회만
         ('Transfer learned', 'Before', OUT + 'pretrain_keep.pt'),
         ('Transfer learned', 'After',  OUT + 'pretrain_drop.pt')]

rows, t0 = [], time.time()
for model, cond, ckpt in CONDS:
    for sd in SEEDS:
        for fi, (tr, te) in enumerate(FOLDS[sd]):
            r = finetune_once(tr, te, yv, ckpt, sd)
            r.update(model=model, cond=cond, seed=sd, fold=fi, n_test=len(te))
            rows.append(r)
            print(f'{model:16s} {cond:6s} s{sd} f{fi}  '
                  f'AUROC {r["auroc"]:.4f}  ({time.time()-t0:.0f}s)', flush=True)

dedup = pd.DataFrame(rows)

# Clinical only 의 After 는 Before 와 동일 (재학습하지 않고 복제)
co = dedup[dedup.model == 'Clinical only'].copy()
co['cond'] = 'After'
dedup = pd.concat([dedup, co], ignore_index=True)

dedup.to_csv(OUT + 'dedup_cv_raw_scaffold.csv', index=False)
print(len(dedup), 'rows')

Clinical only    Before s42 f0  AUROC 0.6061  (36s)
Clinical only    Before s42 f1  AUROC 0.6600  (60s)
Clinical only    Before s42 f2  AUROC 0.6453  (78s)
Clinical only    Before s42 f3  AUROC 0.6539  (95s)
Clinical only    Before s42 f4  AUROC 0.6095  (121s)
Clinical only    Before s43 f0  AUROC 0.6649  (158s)
Clinical only    Before s43 f1  AUROC 0.6350  (187s)
Clinical only    Before s43 f2  AUROC 0.6563  (246s)
Clinical only    Before s43 f3  AUROC 0.6459  (279s)
Clinical only    Before s43 f4  AUROC 0.6489  (316s)
Clinical only    Before s44 f0  AUROC 0.6202  (370s)
Clinical only    Before s44 f1  AUROC 0.6188  (390s)
Clinical only    Before s44 f2  AUROC 0.7031  (437s)
Clinical only    Before s44 f3  AUROC 0.6346  (481s)
Clinical only    Before s44 f4  AUROC 0.5837  (513s)
Transfer learned Before s42 f0  AUROC 0.6526  (544s)
Transfer learned Before s42 f1  AUROC 0.6750  (575s)
Transfer learned Before s42 f2  AUROC 0.6615  (603s)
Transfer learned Before s42 f3  AUROC 0.6742  (625

In [23]:
# ── 셀 7 │ 표 ──
from scipy.stats import wilcoxon, ttest_rel

MET   = ['accuracy','auroc','aucpr','f1','precision','recall']
LABEL = ['Accuracy','AUCROC','AUCPR','f1 score','precision','recall']
MODELS, CONDL = ['Clinical only','Transfer learned'], ['Before','After']

g = dedup.groupby(['model','cond'])[MET].agg(['mean','std'])

out = []
for mdl in MODELS:
    out.append([mdl, ''] + [''] * len(MET))          # 모델 이름만 있는 머리줄
    for cd in CONDL:
        cells = [f'{g.loc[(mdl, cd), (m, "mean")]:.4f} ± '
                 f'{g.loc[(mdl, cd), (m, "std")]:.4f}' for m in MET]
        out.append(['', cd] + cells)

tab = pd.DataFrame(out, columns=['', 'deduplication'] + LABEL)
tab.to_csv(OUT + 'table5_dedup_scaffold.csv', index=False, encoding='utf-8-sig')
print(tab.to_string(index=False))
print('\nScaffold-based CV, mean ± sd over',
      dedup.groupby(["model","cond"]).size().iloc[0], 'folds (3 seeds x 5 folds).')
print('Clinical-only rows are identical by construction: the clinical-only '
      'baseline uses no pre-trained backbone, so removing shared compounds '
      'from the source set cannot affect it.')

print('\n[Transfer learned, After - Before, fold 단위]')
sub = dedup[dedup.model == 'Transfer learned']
b = sub[sub.cond=='Before'].sort_values(['seed','fold'])
a = sub[sub.cond=='After' ].sort_values(['seed','fold'])
assert (b[['seed','fold']].values == a[['seed','fold']].values).all()
for met, lab in zip(MET, LABEL):
    d = a[met].values - b[met].values
    ci = 1.96 * d.std(ddof=1) / np.sqrt(len(d))
    print(f'{lab:10s} {d.mean():+.4f} ± {ci:.4f}  '
          f'wilcoxon p={wilcoxon(b[met], a[met]).pvalue:.4f}  '
          f'paired-t p={ttest_rel(b[met], a[met]).pvalue:.4f}')

                 deduplication        Accuracy          AUCROC           AUCPR        f1 score       precision          recall
   Clinical only                                                                                                              
                        Before 0.6029 ± 0.0339 0.6391 ± 0.0290 0.7310 ± 0.0567 0.6394 ± 0.0571 0.7026 ± 0.0626 0.5979 ± 0.1010
                         After 0.6029 ± 0.0339 0.6391 ± 0.0290 0.7310 ± 0.0567 0.6394 ± 0.0571 0.7026 ± 0.0626 0.5979 ± 0.1010
Transfer learned                                                                                                              
                        Before 0.6157 ± 0.0392 0.6572 ± 0.0332 0.7300 ± 0.0380 0.6446 ± 0.0413 0.7286 ± 0.0556 0.5846 ± 0.0695
                         After 0.6169 ± 0.0426 0.6605 ± 0.0403 0.7326 ± 0.0412 0.6479 ± 0.0387 0.7274 ± 0.0486 0.5894 ± 0.0629

Scaffold-based CV, mean ± sd over 15 folds (3 seeds x 5 folds).
Clinical-only rows are identical by constructi